In [ ]:
#Learn Pydantic 

from pydantic import BaseModel

class Product(BaseModel):
    name:str
    price: float
    tags: list[str]
    metadata: dict[str,str]

iphone = Product( name = "iphone",price = "12.35", tags = ["smartphone","apple"],metadata = {"Country":"India"})    
print(iphone)
print(iphone.name)
print(iphone.price)


name='iphone' price=12.35 tags=['smartphone', 'apple'] metadata={'Country': 'India'}
iphone
12.35


In [ ]:
#Exception Handling in Pydantic 

from pydantic import BaseModel, ValidationError

class ChatRequest(BaseModel):
    message:str
    temperature:float
    model:str


#Automatic type coercion when compataible (e.g, string "0.6" converted to float 0.6)
valid_req = ChatRequest(message="Expain vector db",temperature="0.6",model="Gpt-6")
print(f"Valid Request object created: {valid_req}")
print(f"Temperature type is float {type(valid_req.temperature)}")


#Automatic validation error on incompatible data types

try:
    ChatRequest(message=123,temperature="Good",model="gpt-4")
except ValidationError as e:
    print("\n---------Caught expected Pydantic ValidationError----------")
    print(e)

Valid Request object created: message='Expain vector db' temperature=0.6 model='Gpt-6'
Temperature type is float <class 'float'>

---------Caught expected Pydantic ValidationError----------
2 validation errors for ChatRequest
message
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
temperature
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='Good', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


In [ ]:
# Optional or None in Pydantic 

class ChatRequestOptional(BaseModel):
    message:str                        #Requird field
    temperature:float | None = None    #Optional field defaults to None
    max_tokens: int = 1000             #Optional field defaults to 1000
    system_prompt: str | None          #Explicietly required field that accepts None!


#passing required fields
req1 = ChatRequestOptional(message="Hello",system_prompt=None)
print(req1)


#Failing because 'system_prompt' was ommited even though it accepts None

try:
    ChatRequestOptional(message="hello")
except ValidationError as e:
    print("\n-------Caught error when missing required field 'system_prompt' ---------")
    print(e)


message='Hello' temperature=None max_tokens=1000 system_prompt=None

-------Caught error when missing required field 'system_prompt' ---------
1 validation error for ChatRequestOptional
system_prompt
  Field required [type=missing, input_value={'message': 'hello'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [ ]:
#Composition in Pydantic 

class ModelConfig(BaseModel):
    model:str
    temperature:float

class Message(BaseModel):
    role:str
    content:str

class NestedChatRequest(BaseModel):
    config: ModelConfig
    messages:list[Message]


raw_payload = {
    "config":{"model":"gpt-4o","temperature":20},
    "messages":[
        {"role":"system","content":"you are an assistant"},
        {"role":"user","content":"what is RAG"}
    ]
}


req1 = NestedChatRequest(**raw_payload)  # ** expands the complete payload 
print(req1)
print(req1.config)

config=ModelConfig(model='gpt-4o', temperature=20.0) messages=[Message(role='system', content='you are an assistant'), Message(role='user', content='what is RAG')]
model='gpt-4o' temperature=20.0


In [35]:
#Field in Pydantic 

from pydantic import Field

class ConstrainedChatRequest(BaseModel):
    message:str = Field(min_length=1,max_length=10)
    temperature:float = Field(default=0.7,ge=0.2,le=2.0)
    #Alias maps incoming key 'user_id' to internal variable name 'user_identifier'
    user_identifier :str  = Field(alias="user_id")


#Passing payload using the alias name 'user_id'

req = ConstrainedChatRequest(message="Hello",user_id="usr_1234")
print(req)
print(f"Parsed  Alias -> Internal Attribute: {req.user_identifier}")

message='Hello' temperature=0.7 user_identifier='usr_1234'
Parsed  Alias -> Internal Attribute: usr_1234


In [37]:
#Typing in Pydantic/Python

from typing import Annotated
from pydantic import BaseModel, Field, ValidationError


class ConstrainedChatRequest(BaseModel):
    message:Annotated[str,Field(min_length=1,max_length=10)] = "default"  #Modern way to write using Annotated
    temperature: float = Field(default=0.7,ge=0.2,le=2.1)



#Resuable Custom Domain Types with Annotated
PromptedText = Annotated[
    str,
    Field(min_length=1, max_length=20,description='the primary user prompt')
]

Temperature = Annotated[
    float,
    Field(ge=0.2,le=2.0,description='sampling temperature')
]


#Modern Defination using modern annoted syntax

class ConstrainedChatRequest(BaseModel):
    message:PromptedText
    temperature:Temperature = 0.7

    #Inline Annotated Field using alias mapping
    user_identifier : Annotated[str,Field(alias="user_id")]

req = ConstrainedChatRequest(message="Explain Vector DB's",user_id="usr_1234")
print(f"Parsed  Alias -> Internal Attribute: {req.user_identifier}")
print(f"Default Temperature Applied: {req.temperature}")


Parsed  Alias -> Internal Attribute: usr_1234
Default Temperature Applied: 0.7


In [47]:
from pydantic import field_validator,model_validator

class SignupRequest(BaseModel):
    username:str
    password:str
    confirm_password:str


    #Single Field Transformation/Validation

    @field_validator("username")
    @classmethod
    def validate_username(cls,value:str) -> str:
        if " " in value:
            raise ValueError("Username cannot have spaces")
        return value.lower() #Normalize username to lowercase


    @model_validator(mode="after")
    def verify_password_match(self):
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self


signup = SignupRequest(username="PareshBhamare",password="1234",confirm_password="1234")
print(signup)

username='pareshbhamare' password='1234' confirm_password='1234'


In [63]:
from pydantic import ConfigDict , SecretStr, EmailStr

class ApiConfiguration(BaseModel):
    #Strict  configuration throw error if extra unexpected key is passed or sent    
    model_config = ConfigDict(extra="forbid")

    admin_email:EmailStr
    api_key:SecretStr

config = ApiConfiguration(
    admin_email="admin@ai-service.com",
    api_key = "sk-proj-secret-token-123"
)   

print(config)

admin_email='admin@ai-service.com' api_key=SecretStr('**********')


In [ ]:
raw_dict = {'name':'paresh','age':28,'email':'paresh@innovantes.in'}


class User(BaseModel):
    name:str
    age:int
    email:EmailStr

# user  =  User(**raw_dict) #old way

user = User.model_validate(raw_dict)  #Recommended by Pydantic

user_dict = user.model_dump()
print(user_dict)
print(type(user_dict))

user_json = user.model_dump_json()
print(user_json )
print(type(user_json ))

print(user)


<class 'dict'>
{'name': 'paresh', 'age': 28, 'email': 'paresh@innovantes.in'}
{"name":"paresh","age":28,"email":"paresh@innovantes.in"}
<class 'dict'>
name='paresh' age=28 email='paresh@innovantes.in'
